# Encoding Categorical Data
### Transforming Qualitative Attributes into Mathematically Sound Numerical Representations

---

### Scope & Focus of This Notebook
- **Primary Topic**: Categorical Data Encoding foundations — understanding nominal vs. ordinal data, ordinal encoding, target label encoding, the artificial distance trap, and introductory one-hot encoding.
- **What You Will Learn**:
  1. The core distinction: Nominal (unordered: City, Gender) vs. Ordinal (ordered: Education, Tier).
  2. Why arbitrary integer encoding fails: The artificial distance and hierarchy trap.
  3. `OrdinalEncoder`: Preserving true domain ranks using explicit category lists.
  4. `LabelEncoder`: Why it is strictly reserved for the 1D target label ($y$), NOT input features ($X$).
  5. `OneHotEncoder`: Introducing binary indicator vectors for nominal features.
  6. Practical ML Case Study: Preprocessing Titanic features (`Pclass`, `Sex`, `Embarked`) and feeding them cleanly into `LogisticRegression`.
- **What We Will NOT Cover Here**:
  - The Dummy Variable Trap, multicollinearity proofs, `drop='first'`, `drop='if_binary'`, unseen categories, and high-cardinality rare category grouping are covered in detail in **`04_One_Hot_Encoding`**.
  - Multi-column pipeline orchestration belongs to **`05_Column_Transformer`** and **`06_Machine_Learning_Pipeline`**.

---

## 1. Setup & Environment Verification

### What are we doing?
We import data manipulation, encoding, and modeling libraries.


In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Robust path resolution
data_path = '../../data/titanic.csv' if os.path.exists('../../data/titanic.csv') else 'data/titanic.csv'
df_titanic = pd.read_csv(data_path)

print(f"Loaded Titanic dataset: {df_titanic.shape}")


Loaded Titanic dataset: (891, 12)


## 2. Understanding Categorical Data: Nominal vs. Ordinal

Machine learning models are mathematical functions that perform matrix multiplications and distance calculations. They cannot multiply the word `"female"` by a coefficient $\beta_1$.

Categorical features divide strictly into two types:
1. **Nominal Data (Unordered)**: Categories with no inherent mathematical ranking or sequence.
   - Examples: `Gender` (Male, Female), `City` (Delhi, Mumbai, Chennai), `Embarked` (C, Q, S).
2. **Ordinal Data (Ordered / Ranked)**: Categories with an unambiguous, natural hierarchy or progression.
   - Examples: `Education` (High School < Bachelor's < Master's < PhD), `Customer Tier` (Bronze < Silver < Gold).


In [2]:
toy_df = pd.DataFrame({
    'City': ['Delhi', 'Mumbai', 'Chennai', 'Delhi'],         # Nominal
    'Education': ['High School', 'PhD', 'Bachelor', 'Master'] # Ordinal
})
toy_df


,City,Education
0,Delhi,High School
1,Mumbai,PhD
2,Chennai,Bachelor
3,Delhi,Master


## 3. The Artificial Distance Trap (Why Arbitrary Numbers Fail)

### What happens if we assign naive numbers to Nominal data?
Suppose we map:
- `Chennai` $\implies 0$
- `Delhi` $\implies 1$
- `Mumbai` $\implies 2$

### The Catastrophic ML Implication:
1. **False Hierarchy**: The algorithm assumes: $\text{Mumbai} > \text{Delhi} > \text{Chennai}$.
2. **False Distance**: The distance between Chennai (0) and Mumbai (2) is $2$, but between Delhi (1) and Mumbai (2) is only $1$.
The model believes Mumbai is mathematically "closer" to Delhi than to Chennai! This corrupts linear regressions, neural networks, and distance-based algorithms.


## 4. Ordinal Encoding (`OrdinalEncoder`)

### What are we doing?
We encode the ordinal feature `Education` preserving its true natural order.

#### What does `OrdinalEncoder` do?
Scikit-Learn transformer that converts categorical string ranks into ordered integers ($0, 1, 2, \dots$).

#### Important Rule: Explicit Category Ordering
By default, `OrdinalEncoder()` sorts strings alphabetically (`'Bachelor'` < `'High School'` < `'Master'` < `'PhD'`), which is factually incorrect!
You must **always pass explicit category lists** via `categories=[...]` to reflect genuine domain hierarchy.


In [3]:
# Explicit domain ordering: High School < Bachelor < Master < PhD
edu_order = [['High School', 'Bachelor', 'Master', 'PhD']]

oe = OrdinalEncoder(categories=edu_order)
toy_df['Education_Encoded'] = oe.fit_transform(toy_df[['Education']]).astype(int)

toy_df[['Education', 'Education_Encoded']]


,Education,Education_Encoded
0,High School,0
1,PhD,3
2,Bachelor,1
3,Master,2


## 5. Label Encoding (`LabelEncoder`) — Target ($y$) ONLY!

### What are we doing?
We encode a 1-dimensional categorical target vector using `LabelEncoder`.

#### What does `LabelEncoder` do?
Encodes 1D target labels into integers from $0$ to $k-1$.

#### Critical Scikit-Learn Protocol:
- **`LabelEncoder` is strictly designed for 1D target arrays ($y$)**.
- It does **NOT** accept 2D feature matrices and does not integrate cleanly into `ColumnTransformer`.
- **Never use `LabelEncoder` on input features ($X$)**; use `OrdinalEncoder` or `OneHotEncoder` instead.


In [4]:
target_labels = pd.Series(['Churned', 'Retained', 'Retained', 'Churned'])

le = LabelEncoder()
encoded_target = le.fit_transform(target_labels)

pd.DataFrame({
    'Target Label (y)': target_labels,
    'LabelEncoded (0/1)': encoded_target
})


,Target Label (y),LabelEncoded (0/1)
0,Churned,0
1,Retained,1
2,Retained,1
3,Churned,0


## 6. One-Hot Encoding (`OneHotEncoder`) — The Proper Solution for Nominal Data

### What are we doing?
We encode the nominal `City` feature into binary indicator columns.

#### What does `OneHotEncoder` do?
Creates a new binary column ($0$ or $1$) for each unique categorical level, ensuring every pair of categories is mathematically equidistant.

#### Important Parameters:
- `sparse_output=False`: Returns a standard dense NumPy array instead of a scipy sparse matrix.
- `handle_unknown='ignore'`: Prevents crashes during test/production inference when encountering unseen categories.


In [5]:
ohe = OneHotEncoder(sparse_output=False)
city_ohe = ohe.fit_transform(toy_df[['City']])
city_ohe_df = pd.DataFrame(city_ohe, columns=ohe.get_feature_names_out(['City']))

pd.concat([toy_df[['City']], city_ohe_df], axis=1)


,City,City_Chennai,City_Delhi,City_Mumbai
0,Delhi,0.0,1.0,0.0
1,Mumbai,0.0,0.0,1.0
2,Chennai,1.0,0.0,0.0
3,Delhi,0.0,1.0,0.0


## 7. Practical ML Case Study: Titanic Categorical Preprocessing

### What are we doing?
We preprocess the Titanic dataset by applying:
1. **Ordinal Encoding** to `Pclass` (3rd < 2nd < 1st class hierarchy).
2. **One-Hot Encoding** to `Sex` and `Embarked` (nominal features).
3. Train/Test split **before** fitting to prevent data leakage.
4. Verify that preprocessed data trains a `LogisticRegression` model without error.


In [6]:
# Extract subset and drop missing rows for demonstration
titanic_sub = df_titanic[['Pclass', 'Sex', 'Embarked', 'Survived']].dropna().copy()

# Map numeric Pclass to explicit descriptive strings to demonstrate custom ordinal ranking
titanic_sub['Pclass_Tier'] = titanic_sub['Pclass'].map({3: '3rd_Class', 2: '2nd_Class', 1: '1st_Class'})

X = titanic_sub[['Pclass_Tier', 'Sex', 'Embarked']]
y = titanic_sub['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train samples: {len(X_train)}, X_test samples: {len(X_test)}")


X_train samples: 711, X_test samples: 178


In [7]:
# 1. Ordinal Encoding for Pclass_Tier (3rd < 2nd < 1st class)
pclass_order = [['3rd_Class', '2nd_Class', '1st_Class']]  # 3rd = 0, 2nd = 1, 1st = 2
oe_pclass = OrdinalEncoder(categories=pclass_order)
X_train_pclass = oe_pclass.fit_transform(X_train[['Pclass_Tier']])
X_test_pclass = oe_pclass.transform(X_test[['Pclass_Tier']])

# 2. One-Hot Encoding for Sex and Embarked
ohe_nominal = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_nom = ohe_nominal.fit_transform(X_train[['Sex', 'Embarked']])
X_test_nom = ohe_nominal.transform(X_test[['Sex', 'Embarked']])

# 3. Stack features horizontally into unified feature matrices
X_train_ready = np.hstack([X_train_pclass, X_train_nom])
X_test_ready = np.hstack([X_test_pclass, X_test_nom])

print(f"X_train_ready shape: {X_train_ready.shape}")
print(f"X_test_ready shape:  {X_test_ready.shape}")


X_train_ready shape: (711, 6)
X_test_ready shape:  (178, 6)


### Model Verification: Feeding Encoded Data into Logistic Regression

### What are we doing?
We fit a Logistic Regression classifier on the encoded feature matrix to prove that non-numeric categorical attributes have been transformed into model-ready numbers.


In [8]:
clf = LogisticRegression(random_state=42)
clf.fit(X_train_ready, y_train)

y_pred = clf.predict(X_test_ready)
test_accuracy = accuracy_score(y_test, y_pred)

print(f"Logistic Regression Test Accuracy: {test_accuracy * 100:.2f}%")


Logistic Regression Test Accuracy: 76.40%


/Users/shamvi/ml-env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/shamvi/ml-env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/shamvi/ml-env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/shamvi/ml-env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shamvi/ml-env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shamvi/ml-env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


---

## What We Learned
- **Nominal vs. Ordinal**: Nominal features possess no natural order; Ordinal features have a strict progression.
- **The Artificial Distance Trap**: Assigning sequential integers to nominal features injects false hierarchies and spurious distances that corrupt machine learning models.
- **OrdinalEncoder Protocol**: Always supply explicit ordered category lists via `categories=[...]` rather than accepting arbitrary alphabetical defaults.
- **LabelEncoder vs. OrdinalEncoder**: `LabelEncoder` is strictly for 1D target labels ($y$); `OrdinalEncoder` is for 2D feature matrices ($X$).
- **One-Hot Encoding**: Maps nominal categories to binary indicator columns, preserving mathematical equidistance between all categories.

---

## Important Takeaways
1. **Never use `LabelEncoder` on input features ($X$)**: It is not designed for 2D inputs and corrupts nominal features.
2. **Never leave `OrdinalEncoder` to default sorting**: Alphabetical sorting (`"Bachelor"` < `"Master"` < `"PhD"`) will misalign your hierarchy.
3. **Always set `handle_unknown='ignore'` in `OneHotEncoder`**: Protects production inference pipelines from crashing on unseen category strings.

---

## Common Mistakes
1. **Confusing Nominal and Ordinal**: Encoding unordered features (like `City` or `DeviceType`) with integer ranks teaches linear models false linear correlations.
2. **Fitting encoders on the test set**: Always call `fit_transform()` on `X_train` and `transform()` on `X_test`.
3. **Encoding high-cardinality features with OHE blindly**: Applying OHE to a column with 1,000 unique levels creates 1,000 columns, causing dimensionality explosion.

---

## Final Mental Model

```text
┌────────────────────────────────────────────────────────────────────────┐
│                     CATEGORICAL ENCODING DECISION                      │
└────────────────────────────────────────────────────────────────────────┘
                               │
               Is it an Input Feature (X) or Target (y)?
                               │
            ┌──────────────────┴──────────────────┐
            ▼                                     ▼
     Input Feature (X)                        Target (y)
            │                                     │
   Does it have a natural order?                  ▼
            │                               LabelEncoder()
     ┌──────┴──────┐
     ▼             ▼
    YES            NO
 (Ordinal)      (Nominal)
     │             │
     ▼             ▼
OrdinalEncoder  OneHotEncoder
(with order!)   (sparse_output=False)
```
